In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import json_tricks


In [ ]:
LOW_PRECISION_DTYPE = np.float16
LINALG_DTYPE = np.float32


def to_low_precision(values):
    return np.asarray(values, dtype=LOW_PRECISION_DTYPE)


inputs = json_tricks.load('inputs/inputs.json')
for case in inputs:
    case['X'] = to_low_precision(case['X'])
    case['y'] = to_low_precision(case['y'])
answer = {
    f'case_{index}': {}
    for index, case in enumerate(inputs)
}


# Linear Regression: Geometry, Scaling, and Decorrelation

Linear regression looks simple: choose weights that minimize mean squared error. But the geometry of the feature matrix can make the same optimization problem either friendly or painfully slow.

In this task we will use the Boston housing-price dataset, the classic dataset formerly available as `sklearn.datasets.load_boston`. We store the data and run the arithmetic in `float16` to make the numerical fragility visible. We select features with non-zero means, non-unit variances, and awkward geometry. We will solve linear regression analytically with the explicit normal-equation inverse, trace gradient descent, and then repair the geometry by decorrelating or at least normalizing the features.


# Task 1. Dataset and Mean Squared Error

The dataset has deliberately awkward feature scales. Before solving anything, implement helpers for the linear model and MSE loss.

Implement the functions below:

- `add_bias_column(X)`: return a design matrix with a column of ones inserted before the original feature columns.
- `predict_linear(X_design, weights)`: return the vector of predictions `X_design @ weights`.
- `mse_loss(X_design, weights, y)`: return the mean squared error between predictions and targets.

Keep the computations in the notebook's low-precision dtype so the numerical problems remain visible.


In [ ]:
def add_bias_column(X):
    ## YOUR CODE HERE
    return np.zeros((X.shape[0], X.shape[1] + 1), dtype=LOW_PRECISION_DTYPE)


def predict_linear(X_design, weights):
    ## YOUR CODE HERE
    return np.zeros(X_design.shape[0], dtype=LOW_PRECISION_DTYPE)


def mse_loss(X_design, weights, y):
    ## YOUR CODE HERE
    return LOW_PRECISION_DTYPE(0)


# Check 1. Dataset Shape and Feature Scales

The bars show that the selected features are not centered and do not have unit variance.


In [ ]:
case0 = inputs[0]
X_raw = case0['X']
y_raw = case0['y']
X_design_raw = add_bias_column(X_raw)
two_feature_indices = [0, 3]
zero_weights = np.zeros(X_design_raw.shape[1], dtype=LOW_PRECISION_DTYPE)

np.testing.assert_equal(X_design_raw.shape, (X_raw.shape[0], X_raw.shape[1] + 1))
np.testing.assert_allclose(X_design_raw[:, 0], np.ones(X_raw.shape[0], dtype=LOW_PRECISION_DTYPE))
np.testing.assert_allclose(predict_linear(X_design_raw, zero_weights), np.zeros(X_raw.shape[0], dtype=LOW_PRECISION_DTYPE))
np.testing.assert_allclose(mse_loss(X_design_raw, zero_weights, y_raw), mse_loss(X_design_raw, zero_weights, y_raw))

feature_means = X_raw.mean(axis=0)
feature_stds = X_raw.std(axis=0)

fig, axes = plt.subplots(1, 2, figsize=(9, 3))
axes[0].bar(np.arange(X_raw.shape[1]), feature_means)
axes[0].set_title('feature means')
axes[0].set_xlabel('feature')
axes[1].bar(np.arange(X_raw.shape[1]), feature_stds)
axes[1].set_title('feature standard deviations')
axes[1].set_xlabel('feature')
plt.tight_layout()
plt.show()

for index, case in enumerate(inputs):
    X_design = add_bias_column(case['X'])
    answer[f'case_{index}']['raw_design_shape'] = np.array(X_design.shape)
    answer[f'case_{index}']['raw_feature_means'] = case['X'].mean(axis=0)
    answer[f'case_{index}']['raw_feature_stds'] = case['X'].std(axis=0)
    answer[f'case_{index}']['zero_loss'] = mse_loss(X_design, np.zeros(X_design.shape[1], dtype=LOW_PRECISION_DTYPE), case['y'])

print('feature means:', feature_means)
print('feature stds:', feature_stds)


# Task 2. Normal-Equation Inverse

For linear regression with MSE, the closed-form solution is

$$w^* = (X^T X)^{-1} X^T y,$$

Implement the functions below:

- `normal_equation_solution(X_design, y)`: form the Gram matrix `X_design.T @ X_design`, form the right-hand side `X_design.T @ y`, and return the normal-equation weights.
- `gram_eigenvalues(X_design)`: return the eigenvalues of `X_design.T @ X_design`; these values will be used to diagnose ill-conditioning.

The raw matrix is ill-conditioned, so the weights are a bad numerical representation of the same model even when the loss is low.


In [ ]:
def normal_equation_solution(X_design, y):
    ## YOUR CODE HERE
    return np.zeros(X_design.shape[1], dtype=LOW_PRECISION_DTYPE)


def gram_eigenvalues(X_design):
    ## YOUR CODE HERE
    return np.zeros(X_design.shape[1], dtype=LINALG_DTYPE)


# Check 2. Raw Normal-Equation Weights

The explicit normal-equation inverse reaches a low MSE here, but on raw nitric-oxide, room-count, distance, and highway-access scales the matrix is ill-conditioned. This is what we mean here by bad weights: the inverse amplifies small numerical and data perturbations, so the coefficients are not a comfortable representation.

With `float16`, the damage is no longer just theoretical: the Gram matrix and right-hand side are rounded before the inverse is computed. NumPy still needs a `float32` cast for the final `inv`, but by then the low-precision products have already lost information.

Even the pseudo-inverse is not a magic escape hatch. It is usually much safer than the explicit inverse because it is based on the SVD, but if `X.T @ X` is extremely ill-conditioned, the data are very low precision, or the smallest singular directions still carry target signal, then `np.linalg.pinv(X) @ y` can also return unstable coefficients.


In [ ]:
w_normal_raw = normal_equation_solution(X_design_raw, y_raw)
loss_normal_raw = mse_loss(X_design_raw, w_normal_raw, y_raw)
eig_raw = gram_eigenvalues(X_design_raw)
condition_raw = eig_raw.max() / eig_raw[eig_raw > 1e-10].min()

np.testing.assert_equal(w_normal_raw.shape, (X_design_raw.shape[1],))
np.testing.assert_allclose(loss_normal_raw, np.min([loss_normal_raw, mse_loss(X_design_raw, zero_weights, y_raw)]), atol=1e-2)
np.testing.assert_array_less(0, eig_raw[eig_raw > 1e-10])

plt.figure(figsize=(6, 3))
plt.bar(np.arange(len(w_normal_raw)), w_normal_raw)
plt.axhline(0, color='black', linewidth=1)
plt.xlabel('coefficient index, including bias')
plt.ylabel('weight value')
plt.title('raw normal-equation weights')
plt.show()

for index, case in enumerate(inputs):
    X_design = add_bias_column(case['X'])
    w = normal_equation_solution(X_design, case['y'])
    eig = gram_eigenvalues(X_design)
    positive = eig[eig > 1e-10]
    answer[f'case_{index}']['raw_normal_equation_weights'] = w
    answer[f'case_{index}']['raw_normal_equation_loss'] = mse_loss(X_design, w, case['y'])
    answer[f'case_{index}']['raw_gram_eigenvalues'] = eig
    answer[f'case_{index}']['raw_condition_number'] = positive.max() / positive.min()

print('raw normal-equation weights:', w_normal_raw)
print('raw normal-equation loss:', loss_normal_raw)
print('condition number of X.T @ X:', condition_raw)


# Task 3. Two-Feature Loss Geometry and Gradient Descent

Now keep only features 0 and 3. We will draw the loss surface over the two weights while the bias is fixed to its normal-equation value. The contours are long and narrow because the matrix $X^T X$ has eigenvalues with very different sizes.

Implement the functions below:

- `gradient_of_mse(X_design, weights, y)`: return the gradient of MSE with respect to the weights, using `2 * X_design.T @ residual / n`.
- `gradient_descent(X_design, y, initial_weights, learning_rate, n_steps)`: repeatedly update the weights by subtracting `learning_rate * gradient`, and return both the full weight history and the loss history.

The returned history arrays should include the initial weights and initial loss before any update.


In [ ]:
def gradient_of_mse(X_design, weights, y):
    ## YOUR CODE HERE
    return np.zeros_like(weights, dtype=LOW_PRECISION_DTYPE)


def gradient_descent(X_design, y, initial_weights, learning_rate, n_steps):
    ## YOUR CODE HERE
    return np.zeros((n_steps + 1, initial_weights.shape[0]), dtype=LOW_PRECISION_DTYPE), np.zeros(n_steps + 1, dtype=LOW_PRECISION_DTYPE)


In [ ]:
def loss_grid_for_two_weights(X_two_design, y, bias, w1_range, w2_range):
    W1, W2 = np.meshgrid(w1_range, w2_range)
    losses = np.zeros_like(W1)
    for row in range(W1.shape[0]):
        for col in range(W1.shape[1]):
            weights = np.array([bias, W1[row, col], W2[row, col]])
            losses[row, col] = mse_loss(X_two_design, weights, y)
    return W1, W2, losses


def plot_two_weight_geometry(X_two_design, y, optimum, history, title):
    span1 = max(5.0, abs(optimum[1]) * 0.9)
    span2 = max(5.0, abs(optimum[2]) * 0.9)
    w1_range = np.linspace(optimum[1] - span1, optimum[1] + span1, 100)
    w2_range = np.linspace(optimum[2] - span2, optimum[2] + span2, 100)
    W1, W2, losses = loss_grid_for_two_weights(X_two_design, y, optimum[0], w1_range, w2_range)

    plt.figure(figsize=(6, 5))
    levels = np.geomspace(max(losses.min(), 1e-8), losses.max(), 22)
    plt.contour(W1, W2, losses, levels=levels)
    plt.scatter([optimum[1]], [optimum[2]], color='tab:red', label='normal-equation optimum')
    plt.plot(history[:, 1], history[:, 2], color='tab:orange', marker='o', markersize=2, linewidth=1, label='gradient descent')
    plt.xlabel('weight for feature 1')
    plt.ylabel('weight for feature 2')
    plt.title(title)
    plt.legend()
    plt.show()


# Check 3. Raw Two-Feature Contours

The theoretical solution is off-center in this plot because the raw features have large means. Gradient descent moves slowly along the flat direction of the valley.


In [ ]:
X_two_raw = X_raw[:, two_feature_indices]
X_two_design_raw = add_bias_column(X_two_raw)
w_two_raw = normal_equation_solution(X_two_design_raw, y_raw)
eig_two_raw = gram_eigenvalues(X_two_design_raw)
learning_rate_raw = 0.8 / (2 * eig_two_raw.max() / X_two_design_raw.shape[0])
history_raw, loss_history_raw = gradient_descent(
    X_two_design_raw,
    y_raw,
    np.zeros(X_two_design_raw.shape[1], dtype=LOW_PRECISION_DTYPE),
    learning_rate_raw,
    160,
)

np.testing.assert_equal(history_raw.shape, (161, 3))
np.testing.assert_array_less(loss_history_raw[-1], loss_history_raw[0])

plot_two_weight_geometry(X_two_design_raw, y_raw, w_two_raw, history_raw, 'raw correlated features')
plt.figure(figsize=(6, 3))
plt.plot(loss_history_raw)
plt.yscale('log')
plt.xlabel('gradient descent step')
plt.ylabel('MSE')
plt.title('raw two-feature loss history')
plt.show()

for index, case in enumerate(inputs):
    X_two = case['X'][:, two_feature_indices]
    X_design = add_bias_column(X_two)
    eig = gram_eigenvalues(X_design)
    lr = 0.8 / (2 * eig.max() / X_design.shape[0])
    hist, losses = gradient_descent(X_design, case['y'], np.zeros(X_design.shape[1], dtype=LOW_PRECISION_DTYPE), lr, 160)
    answer[f'case_{index}']['raw_two_feature_normal_equation_weights'] = normal_equation_solution(X_design, case['y'])
    answer[f'case_{index}']['raw_two_feature_gd_final_weights'] = hist[-1]
    answer[f'case_{index}']['raw_two_feature_loss_history'] = losses
    answer[f'case_{index}']['raw_two_feature_eigenvalues'] = eig

print('raw two-feature eigenvalues:', eig_two_raw)
print('raw two-feature final GD loss:', loss_history_raw[-1])


# Why Is Raw Gradient Descent Slow?

For MSE, the curvature of the loss is controlled by $X^T X$. Its eigenvectors are the main axes of the quadratic bowl, and its eigenvalues say how steep the bowl is along those axes.

When one eigenvalue is huge and another is tiny, the contours become a long narrow valley. A learning rate small enough not to explode in the steep direction is too small to move quickly in the flat direction. This is why raw correlated features make gradient descent crawl.


# Task 4. Decorrelation by Whitening

Decorrelation rotates and rescales the centered features so the covariance matrix becomes close to identity. In this coordinate system the loss contours are much rounder.

Implement `decorrelate_features(X)` so that it returns three objects:

- `X_decorrelated`: centered and whitened features.
- `mean`: the row vector of feature means used for centering.
- `whitening`: the whitening matrix used to transform centered data.

Use the covariance eigen-decomposition to build the whitening matrix, and keep the result in the notebook's low-precision dtype.


In [ ]:
def decorrelate_features(X):
    ## YOUR CODE HERE
    return np.zeros_like(X, dtype=LOW_PRECISION_DTYPE), np.zeros((1, X.shape[1]), dtype=LOW_PRECISION_DTYPE), np.eye(X.shape[1], dtype=LOW_PRECISION_DTYPE)


# Check 4. Decorrelated Two-Feature Optimization

The normal-equation optimum moves close to the center of the contour plot, and gradient descent reaches the minimum much faster.


In [ ]:
X_two_decorrelated, mean_two, whitening_two = decorrelate_features(X_two_raw)
X_two_design_decorrelated = add_bias_column(X_two_decorrelated)
w_two_decorrelated = normal_equation_solution(X_two_design_decorrelated, y_raw)
eig_two_decorrelated = gram_eigenvalues(X_two_design_decorrelated)
learning_rate_decorrelated = 0.8 / (2 * eig_two_decorrelated.max() / X_two_design_decorrelated.shape[0])
history_decorrelated, loss_history_decorrelated = gradient_descent(
    X_two_design_decorrelated,
    y_raw,
    np.zeros(X_two_design_decorrelated.shape[1], dtype=LOW_PRECISION_DTYPE),
    learning_rate_decorrelated,
    40,
)

np.testing.assert_allclose(X_two_decorrelated.mean(axis=0), np.zeros(2), atol=5e-3)
np.testing.assert_allclose(np.cov(X_two_decorrelated, rowvar=False, bias=True), np.eye(2), atol=6e-2)
np.testing.assert_array_less(loss_history_decorrelated[-1], loss_history_raw[-1])

plot_two_weight_geometry(X_two_design_decorrelated, y_raw, w_two_decorrelated, history_decorrelated, 'decorrelated features')
plt.figure(figsize=(6, 3))
plt.plot(loss_history_raw[:41], label='raw')
plt.plot(loss_history_decorrelated, label='decorrelated')
plt.yscale('log')
plt.xlabel('gradient descent step')
plt.ylabel('MSE')
plt.title('decorrelation speeds up convergence')
plt.legend()
plt.show()

for index, case in enumerate(inputs):
    X_two = case['X'][:, two_feature_indices]
    X_decorrelated, mean, whitening = decorrelate_features(X_two)
    X_design = add_bias_column(X_decorrelated)
    eig = gram_eigenvalues(X_design)
    lr = 0.8 / (2 * eig.max() / X_design.shape[0])
    hist, losses = gradient_descent(X_design, case['y'], np.zeros(X_design.shape[1], dtype=LOW_PRECISION_DTYPE), lr, 40)
    answer[f'case_{index}']['decorrelated_two_feature_normal_equation_weights'] = normal_equation_solution(X_design, case['y'])
    answer[f'case_{index}']['decorrelated_two_feature_gd_final_weights'] = hist[-1]
    answer[f'case_{index}']['decorrelated_two_feature_loss_history'] = losses
    answer[f'case_{index}']['decorrelated_two_feature_eigenvalues'] = eig
    answer[f'case_{index}']['decorrelated_covariance'] = np.cov(X_decorrelated, rowvar=False, bias=True)

print('decorrelated eigenvalues:', eig_two_decorrelated)
print('decorrelated final GD loss:', loss_history_decorrelated[-1])


# Task 5. Poor Man's Decorrelation: Centering and Normalization

Whitening is very effective, but a simpler baseline often works well: subtract the mean and divide by the standard deviation feature by feature. This does not remove all correlation, but it fixes the most painful scale problem.

Implement `normalize_features(X)` so that it returns three objects:

- `X_normalized`: each feature centered and divided by its own standard deviation.
- `mean`: the row vector of feature means.
- `std`: the row vector of feature standard deviations.

Add a small value to the denominator so division stays safe in low precision.


In [ ]:
def normalize_features(X):
    ## YOUR CODE HERE
    return np.zeros_like(X, dtype=LOW_PRECISION_DTYPE), np.zeros((1, X.shape[1]), dtype=LOW_PRECISION_DTYPE), np.ones((1, X.shape[1]), dtype=LOW_PRECISION_DTYPE)


# Check 5. Normalized Two-Feature Optimization

Even without full whitening, centering and normalization make the contours easier and gradient descent behaves well.


In [ ]:
X_two_normalized, mean_norm_two, std_norm_two = normalize_features(X_two_raw)
X_two_design_normalized = add_bias_column(X_two_normalized)
w_two_normalized = normal_equation_solution(X_two_design_normalized, y_raw)
eig_two_normalized = gram_eigenvalues(X_two_design_normalized)
learning_rate_normalized = 0.8 / (2 * eig_two_normalized.max() / X_two_design_normalized.shape[0])
history_normalized, loss_history_normalized = gradient_descent(
    X_two_design_normalized,
    y_raw,
    np.zeros(X_two_design_normalized.shape[1], dtype=LOW_PRECISION_DTYPE),
    learning_rate_normalized,
    80,
)

np.testing.assert_allclose(X_two_normalized.mean(axis=0), np.zeros(2), atol=5e-3)
np.testing.assert_allclose(X_two_normalized.std(axis=0), np.ones(2), atol=1e-2)
np.testing.assert_array_less(loss_history_normalized[-1], loss_history_raw[-1])

plot_two_weight_geometry(X_two_design_normalized, y_raw, w_two_normalized, history_normalized, 'centered and normalized features')
plt.figure(figsize=(6, 3))
plt.plot(loss_history_raw[:81], label='raw')
plt.plot(loss_history_normalized, label='normalized')
plt.yscale('log')
plt.xlabel('gradient descent step')
plt.ylabel('MSE')
plt.title('normalization is already enough here')
plt.legend()
plt.show()

for index, case in enumerate(inputs):
    X_two = case['X'][:, two_feature_indices]
    X_normalized, mean, std = normalize_features(X_two)
    X_design = add_bias_column(X_normalized)
    eig = gram_eigenvalues(X_design)
    lr = 0.8 / (2 * eig.max() / X_design.shape[0])
    hist, losses = gradient_descent(X_design, case['y'], np.zeros(X_design.shape[1], dtype=LOW_PRECISION_DTYPE), lr, 80)
    answer[f'case_{index}']['normalized_two_feature_normal_equation_weights'] = normal_equation_solution(X_design, case['y'])
    answer[f'case_{index}']['normalized_two_feature_gd_final_weights'] = hist[-1]
    answer[f'case_{index}']['normalized_two_feature_loss_history'] = losses
    answer[f'case_{index}']['normalized_two_feature_eigenvalues'] = eig

print('normalized eigenvalues:', eig_two_normalized)
print('normalized final GD loss:', loss_history_normalized[-1])


# Task 6. Return to All Features

Finally, normalize every feature and fit the full linear regression model. No new function is required here: reuse `normalize_features`, `add_bias_column`, `normal_equation_solution`, and `gradient_descent` on all columns of `X`.

The practical rule of thumb is: for linear regression and gradient descent, decorrelate features if you can, and at least center and normalize them.


In [ ]:
X_all_normalized, mean_all, std_all = normalize_features(X_raw)
X_all_design_normalized = add_bias_column(X_all_normalized)
w_all_normalized = normal_equation_solution(X_all_design_normalized, y_raw)
eig_all_normalized = gram_eigenvalues(X_all_design_normalized)
learning_rate_all = 0.8 / (2 * eig_all_normalized.max() / X_all_design_normalized.shape[0])
history_all_normalized, loss_history_all_normalized = gradient_descent(
    X_all_design_normalized,
    y_raw,
    np.zeros(X_all_design_normalized.shape[1], dtype=LOW_PRECISION_DTYPE),
    learning_rate_all,
    240,
)

np.testing.assert_allclose(X_all_normalized.mean(axis=0), np.zeros(X_raw.shape[1]), atol=5e-3)
np.testing.assert_allclose(X_all_normalized.std(axis=0), np.ones(X_raw.shape[1]), atol=1e-2)
np.testing.assert_allclose(loss_history_all_normalized[-1], mse_loss(X_all_design_normalized, w_all_normalized, y_raw), atol=5e-1)

plt.figure(figsize=(6, 3))
plt.plot(loss_history_all_normalized)
plt.yscale('log')
plt.xlabel('gradient descent step')
plt.ylabel('MSE')
plt.title('all normalized features')
plt.show()

for index, case in enumerate(inputs):
    X_normalized, mean, std = normalize_features(case['X'])
    X_design = add_bias_column(X_normalized)
    eig = gram_eigenvalues(X_design)
    lr = 0.8 / (2 * eig.max() / X_design.shape[0])
    hist, losses = gradient_descent(X_design, case['y'], np.zeros(X_design.shape[1], dtype=LOW_PRECISION_DTYPE), lr, 240)
    w = normal_equation_solution(X_design, case['y'])
    answer[f'case_{index}']['all_normalized_normal_equation_weights'] = w
    answer[f'case_{index}']['all_normalized_normal_equation_loss'] = mse_loss(X_design, w, case['y'])
    answer[f'case_{index}']['all_normalized_gd_final_weights'] = hist[-1]
    answer[f'case_{index}']['all_normalized_loss_history'] = losses
    answer[f'case_{index}']['all_normalized_eigenvalues'] = eig

print('all-feature normalized normal-equation loss:', mse_loss(X_all_design_normalized, w_all_normalized, y_raw))
print('all-feature normalized final GD loss:', loss_history_all_normalized[-1])


# Takeaway

Raw linear regression can have a perfectly valid mathematical optimum and still be a poor optimization problem. The eigenvalues of $X^T X$ reveal the geometry: very different eigenvalues mean long narrow contours and slow gradient descent.

Decorrelation fixes the geometry directly. Centering and normalization are a cheaper approximation, and in many practical cases they are enough. For linear regression to work reliably with gradient-based optimization, decorrelate the features when possible, and at least normalize them.


In [ ]:
json_tricks.dump(answer, '.answer.json')
